# Imports/Settings

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# 1. Стандартная библиотека
import sys
import warnings
from pathlib import Path

# 2. Сторонние библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from hydra import initialize, compose

# 3. Локальные модули
sys.path.append(str(Path.cwd().parent))
from src.core.data import UniversalDataLoader

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
with initialize(version_base=None, config_path="../configs"):
    cfg = compose(config_name="config")

print(f"Проект: {cfg.project_name} | Режим: Feature Engineering")

# --- НАСТРОЙКИ ВИЗУАЛИЗАЦИИ ---
try:
    p_cfg = cfg.logging.plots
    plt.style.use(p_cfg.style)
    plt.rcParams.update({
        'figure.figsize': p_cfg.fig_size,
        'figure.dpi': p_cfg.dpi,
        'font.size': p_cfg.font_size,
        'axes.grid': p_cfg.grid,
        'axes.spines.top': p_cfg.spines_top,
        'axes.spines.right': p_cfg.spines_right
    })
    PLOT_ALPHA = p_cfg.alpha
except AttributeError:
    PLOT_ALPHA = 0.5
    print("Используются дефолтные стили Matplotlib.")

Проект: outsource_project_name | Режим: Feature Engineering


# Data Loading

In [ ]:
loader = UniversalDataLoader(cfg)

df = loader.load_data()
target = cfg.data.tabular.target_col

print(f"Размер датасета: {df.shape}")
df.head(3)

# Feature Generation

In [ ]:
# Создаем копию, чтобы не сломать исходный датасет при экспериментах
df_features = df.copy()

# Гипотеза 1: Влияет ли день недели и час визита?
# Предполагаем, что колонка visitStartTime - это timestamp
if 'visitStartTime' in df_features.columns:
    dt_series = pd.to_datetime(df_features['visitStartTime'], unit='s')
    df_features['visit_hour'] = dt_series.dt.hour
    df_features['visit_dayofweek'] = dt_series.dt.dayofweek
    df_features['is_weekend'] = df_features['visit_dayofweek'].isin([5, 6]).astype(int)

# Гипотеза 2: Категоризация активности по хитам
# Мы уже собрали total_hits в лоадере, давайте сделаем биннинг
df_features['is_high_activity'] = (df_features['total_hits'] > 10).astype(int)

# Гипотеза 3: Взаимодействие девайса и браузера (Комбинированная фича)
if 'device_category' in df_features.columns and 'browser' in df_features.columns:
    df_features['device_browser'] = df_features['device_category'] + "_" + df_features['browser']

print("Новые фичи успешно сгенерированы!")

# Validation

In [ ]:
# Проверим, как 'is_high_activity' влияет на таргет
plt.figure()
sns.barplot(
    data=df_features,
    x='is_high_activity',
    y=target,
    alpha=PLOT_ALPHA
)
plt.title("Конверсия в зависимости от активности (total_hits > 10)")
plt.ylabel("Доля целевых действий")
plt.show()

# Оценка силы признака через Mutual Information (для численных фичей)
numeric_features = ['visit_hour', 'visit_dayofweek', 'total_hits']
# Оставляем только те колонки, которые реально есть в датасете, и удаляем NaN
valid_cols = [c for c in numeric_features if c in df_features.columns]

if valid_cols:
    temp_df = df_features[valid_cols + [target]].dropna()
    mi_scores = mutual_info_classif(temp_df[valid_cols], temp_df[target], random_state=cfg.seed)

    mi_series = pd.Series(mi_scores, index=valid_cols).sort_values(ascending=False)

    plt.figure(figsize=(cfg.logging.plots.fig_size[0], 4))
    mi_series.plot(kind='bar', color='teal', alpha=PLOT_ALPHA)
    plt.title("Mutual Information Scores (Сила связи с таргетом)")
    plt.ylabel("MI Score")
    plt.xticks(rotation=45)
    plt.show()